# **Durability — PCE validation plots**

## **1. Libraries**

In [1]:
%matplotlib inline
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({
                        'font.family': 'serif',
                        'mathtext.fontset': 'cm',
                        'axes.unicode_minus': False
                    })
import matplotlib.ticker as ticker

from functions import *

C:\git-projetos\2024-1_victor_hugo_renata_maria\.venv\Lib\site-packages\UQpy\__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## **2. Config**

`n_latent_samples`, `times`, `installation_year`, `co2_scenario`, `cement_type` and `exposure_conditions`
must match [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb) /
[`02_train_pce.ipynb`](02_train_pce.ipynb) — together they name the files being loaded.

In [2]:
n_latent_samples    = 2500      # must match stage 1/2 — it names the files being loaded
installation_year   = 1980      # must match stage 1/2
co2_scenario        = "SSP2-4.5"  # must match stage 1/2
cement_type         = 3         # must match stage 1/2
exposure_conditions = 2         # must match stage 1/2
times                = np.linspace(0, 100, 5, endpoint=True)   # must match stage 1/2

fig_size   = (5, 4)      # size of each individual figure, in inches
fig_format = 'png'       # format each figure is saved in ('pdf', 'png', ...)
fig_dpi    = 300         # resolution the figure is saved at (dots per inch)

label_fontsize = 14   # font size of the axis labels
tick_fontsize  = 12   # font size of the tick numbers

## **3. Emulator efficiency (speed-up)**

Loads each time step's PCE metamodel and `dataset_unique_train` fresh, and times a prediction on
the training points against the emulator cost recorded by
[`01_generate_dataset.ipynb`](01_generate_dataset.ipynb).

In [3]:
with open(f'{n_latent_samples}_emulator_timing_durability.pkl', 'rb') as f:
    emulator_timing = dill.load(f)

speedup_rows = []
for t in times:
    tag = f'{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}_co2_{co2_scenario}'
    with open(f'{n_latent_samples}_pce_metamodel_{tag}.pkl', 'rb') as f:
        pce_metamodel = dill.load(f)
    with open(f'{n_latent_samples}_dataset_unique_train_{tag}.pkl', 'rb') as f:
        df_unique_train = dill.load(f)
    x_train = df_unique_train[['fck', 'rh', 'cov']].to_numpy()

    emulator_s = float(emulator_timing.loc[emulator_timing['Time (years)'] == t, 'Train total (s)'].iloc[0])

    t_start = time.perf_counter()
    pce_metamodel.predict(x_train)
    surrogate_s = time.perf_counter() - t_start

    speedup_rows.append({
                            'Time (years)':  t,
                            'Emulator (s)':  emulator_s,
                            'Surrogate (s)': surrogate_s,
                            'Speed-up':      emulator_s / surrogate_s,
                        })

speedup = pd.DataFrame(speedup_rows)
print(f"Median speed-up: {speedup['Speed-up'].median():,.0f}x")
speedup

Median speed-up: 11,045x


   Time (years)  Emulator (s)  Surrogate (s)      Speed-up
0           0.0     58.051825       0.005256  11045.288120
1          25.0     26.753293       0.001095  24423.309477
2          50.0     22.773109       0.001799  12660.871468
3          75.0     25.572338       0.002333  10962.079262
4         100.0     21.726070       0.002213   9818.361216

### 3.1 Speed-up over time

In [4]:
fig, ax = plt.subplots(figsize=fig_size)
ax.plot(speedup['Time (years)'], speedup['Speed-up'], marker='o', color='0.25')
ax.set_xlabel('$t$ (years)', fontsize=label_fontsize)
ax.set_ylabel('Speed-up (emulator / surrogate)', fontsize=label_fontsize)
ax.set_yscale('log')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f'{v:,.0f}x'))
ax.tick_params(axis='both', labelsize=tick_fontsize)
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()

fig.savefig(f'{n_latent_samples}_speedup_durability.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
plt.show()

## **4. PCE validation via KL divergence**

Spot-checks one design point against the PCE, straight against the **raw** Monte Carlo `g` samples
of that point (`dataset_full`), with no GLD fitted to them first. The reference density is a KDE of
the raw data; the PCE side is the GLD built from the PCE's `lambda 1`/`lambda 2` plus the
`lambda3`/`lambda4` written by hand below (the mean over the stage-1 dataset — the PCE's own fit for
those two is unreliable, same convention as the benchmark).

`time_index` and `design_point_index` pick which point to check, both by position — so they're
guaranteed to exist in the saved dataset.

In [5]:
time_index         = 2   # index into `times` — which time step's PCE to check
design_point_index = 0   # row index into that time step's dataset_unique_train — which (fck, rh, cov) to check

lambda3 = 0.132011   # written by hand — mean of lambda 3 over the stage-1 dataset
lambda4 = 0.141281   # written by hand — mean of lambda 4 over the stage-1 dataset

n_grid = 400   # grid points used to numerically integrate the KL divergence and R²

xlim = None   # e.g. (-5, 30) to fix the g axis; None = auto-scaled to the data
ylim = None   # e.g. (0, 1) to fix the density axis; None = auto-scaled to the data

In [6]:
t_check = times[time_index]
tag_check = f'{t_check}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}_co2_{co2_scenario}'

with open(f'{n_latent_samples}_dataset_full_train_{tag_check}.pkl', 'rb') as f:
    df_full_check = dill.load(f)
with open(f'{n_latent_samples}_dataset_unique_train_{tag_check}.pkl', 'rb') as f:
    df_unique_check = dill.load(f)
with open(f'{n_latent_samples}_pce_metamodel_{tag_check}.pkl', 'rb') as f:
    pce_metamodel = dill.load(f)

design_point = df_unique_check.iloc[design_point_index]
fck_check, rh_check, cov_check = design_point['fck'], design_point['rh'], design_point['cov']

g_real = df_full_check.loc[(df_full_check['fck'] == fck_check) & (df_full_check['rh'] == rh_check) & (df_full_check['cov'] == cov_check), 'g'].to_numpy()

print(f"Checking t = {t_check:.2f} years, fck = {fck_check:.3f}, rh = {rh_check:.3f}, cov = {cov_check:.3f} ({len(g_real)} raw g samples)")

kl_result = validate_pce_kl_divergence_durability(
                                                     pce_metamodel=pce_metamodel,
                                                     fck=fck_check,
                                                     rh=rh_check,
                                                     cov=cov_check,
                                                     g_real=g_real,
                                                     lambda3=lambda3,
                                                     lambda4=lambda4,
                                                     n_grid=n_grid,
                                                   )

print(f"KL divergence:  {kl_result['kl_divergence']:.5f}")
print(f"KS statistic:   {kl_result['ks_statistic']:.5f}")
print(f"Wasserstein:    {kl_result['wasserstein']:.5f}")
print(f"R2 (PDFs):      {kl_result['r2_pdf']:.5f}")
print(f"Rel. error P5:  {kl_result['rel_err_p5']:+.2%}")
print(f"Rel. error P50: {kl_result['rel_err_p50']:+.2%}")
print(f"Rel. error P95: {kl_result['rel_err_p95']:+.2%}")

Checking t = 50.00 years, fck = 35.146, rh = 56.814, cov = 19.355 (2500 raw g samples)
KL divergence:  0.00724
KS statistic:   0.02720
Wasserstein:    0.02167
R2 (PDFs):      0.99213
Rel. error P5:  -1.08%
Rel. error P50: -1.51%
Rel. error P95: +0.71%


### 4.1 Chart in english

In [7]:
pkl_name = f'{n_latent_samples}_dataset_unique_train_{tag_check}'

fig, ax = plt.subplots(figsize=fig_size)
ax.plot(kl_result['x_grid'], kl_result['pdf_real'], color='0.25', linewidth=2, label='Dataset')
ax.plot(kl_result['x_grid'], kl_result['pdf_pce'], color='crimson', linewidth=2, linestyle='--', label='PCE-predicted GLD')
ax.set_xlabel('$g$', fontsize=label_fontsize)
ax.set_ylabel('Probability density', fontsize=label_fontsize)
ax.tick_params(axis='both', labelsize=tick_fontsize)
if xlim is not None:
    ax.set_xlim(xlim)
if ylim is not None:
    ax.set_ylim(ylim)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.18), ncol=2, fontsize=tick_fontsize, frameon=False)
ax.grid(True, alpha=0.3)
fig.tight_layout()

fig.savefig(f'{pkl_name}_kl_divergence_fck{fck_check:g}_rh{rh_check:g}_cov{cov_check:g}_{t_check}_en.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
plt.show()

### 4.2 Chart in portuguese

In [8]:
fig, ax = plt.subplots(figsize=fig_size)
ax.plot(kl_result['x_grid'], kl_result['pdf_real'], color='0.25', linewidth=2, label='Dados')
ax.plot(kl_result['x_grid'], kl_result['pdf_pce'], color='crimson', linewidth=2, linestyle='--', label='GLD previsto pelo PCE')
ax.set_xlabel('$g$', fontsize=label_fontsize)
ax.set_ylabel('Densidade de probabilidade', fontsize=label_fontsize)
ax.tick_params(axis='both', labelsize=tick_fontsize)
if xlim is not None:
    ax.set_xlim(xlim)
if ylim is not None:
    ax.set_ylim(ylim)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.18), ncol=2, fontsize=tick_fontsize, frameon=False)
ax.grid(True, alpha=0.3)
fig.tight_layout()

fig.savefig(f'{pkl_name}_kl_divergence_fck{fck_check:g}_rh{rh_check:g}_cov{cov_check:g}_{t_check}_pt.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
plt.show()

### 4.3 Second time check — same $(f_{ck}, RH, c)$, another $t$

Repeats section 4 at a **second** time step, on the **same** design point — the design samples are
drawn once in [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb) and reused at every time
step, so `design_point_index` lands on the same design point here as in section 4.

In [9]:
time_index_b = 4   # index into `times` — the second time step to check (section 4 used `time_index`)

lambda3_b = lambda3   # same hand-written lambda 3 / lambda 4 as section 4
lambda4_b = lambda4

In [10]:
t_check_b = times[time_index_b]
tag_check_b = f'{t_check_b}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}_co2_{co2_scenario}'

with open(f'{n_latent_samples}_dataset_full_train_{tag_check_b}.pkl', 'rb') as f:
    df_full_check_b = dill.load(f)
with open(f'{n_latent_samples}_dataset_unique_train_{tag_check_b}.pkl', 'rb') as f:
    df_unique_check_b = dill.load(f)
with open(f'{n_latent_samples}_pce_metamodel_{tag_check_b}.pkl', 'rb') as f:
    pce_metamodel_b = dill.load(f)

design_point_b = df_unique_check_b.iloc[design_point_index]
fck_check_b, rh_check_b, cov_check_b = design_point_b['fck'], design_point_b['rh'], design_point_b['cov']

if not (np.isclose(fck_check_b, fck_check) and np.isclose(rh_check_b, rh_check) and np.isclose(cov_check_b, cov_check)):
    print(f"WARNING: design point {design_point_index} is not the same at both times")

g_real_b = df_full_check_b.loc[(df_full_check_b['fck'] == fck_check_b) & (df_full_check_b['rh'] == rh_check_b) & (df_full_check_b['cov'] == cov_check_b), 'g'].to_numpy()

print(f"Checking t = {t_check_b:.2f} years, fck = {fck_check_b:.3f}, rh = {rh_check_b:.3f}, cov = {cov_check_b:.3f} ({len(g_real_b)} raw g samples)")

kl_result_b = validate_pce_kl_divergence_durability(
                                                       pce_metamodel=pce_metamodel_b,
                                                       fck=fck_check_b,
                                                       rh=rh_check_b,
                                                       cov=cov_check_b,
                                                       g_real=g_real_b,
                                                       lambda3=lambda3_b,
                                                       lambda4=lambda4_b,
                                                       n_grid=n_grid,
                                                     )

print(f"KL divergence:  {kl_result_b['kl_divergence']:.5f}")
print(f"KS statistic:   {kl_result_b['ks_statistic']:.5f}")
print(f"Wasserstein:    {kl_result_b['wasserstein']:.5f}")
print(f"R2 (PDFs):      {kl_result_b['r2_pdf']:.5f}")
print(f"Rel. error P5:  {kl_result_b['rel_err_p5']:+.2%}")
print(f"Rel. error P50: {kl_result_b['rel_err_p50']:+.2%}")
print(f"Rel. error P95: {kl_result_b['rel_err_p95']:+.2%}")

Checking t = 100.00 years, fck = 35.146, rh = 56.814, cov = 19.355 (2500 raw g samples)
KL divergence:  0.00933
KS statistic:   0.02680
Wasserstein:    0.04299
R2 (PDFs):      0.99230
Rel. error P5:  -0.83%
Rel. error P50: -0.92%
Rel. error P95: +1.89%


### 4.4 Chart in english — second time

In [11]:
pkl_name_b = f'{n_latent_samples}_dataset_unique_train_{tag_check_b}'

fig, ax = plt.subplots(figsize=fig_size)
ax.plot(kl_result_b['x_grid'], kl_result_b['pdf_real'], color='0.25', linewidth=2, label='Dataset')
ax.plot(kl_result_b['x_grid'], kl_result_b['pdf_pce'], color='crimson', linewidth=2, linestyle='--', label='PCE-predicted GLD')
ax.set_xlabel('$g$', fontsize=label_fontsize)
ax.set_ylabel('Probability density', fontsize=label_fontsize)
ax.tick_params(axis='both', labelsize=tick_fontsize)
if xlim is not None:
    ax.set_xlim(xlim)
if ylim is not None:
    ax.set_ylim(ylim)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.18), ncol=2, fontsize=tick_fontsize, frameon=False)
ax.grid(True, alpha=0.3)
fig.tight_layout()

fig.savefig(f'{pkl_name_b}_kl_divergence_fck{fck_check_b:g}_rh{rh_check_b:g}_cov{cov_check_b:g}_{t_check_b}_en.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
plt.show()

### 4.5 Chart in portuguese — second time

In [12]:
fig, ax = plt.subplots(figsize=fig_size)
ax.plot(kl_result_b['x_grid'], kl_result_b['pdf_real'], color='0.25', linewidth=2, label='Dados')
ax.plot(kl_result_b['x_grid'], kl_result_b['pdf_pce'], color='crimson', linewidth=2, linestyle='--', label='GLD previsto pelo PCE')
ax.set_xlabel('$g$', fontsize=label_fontsize)
ax.set_ylabel('Densidade de probabilidade', fontsize=label_fontsize)
ax.tick_params(axis='both', labelsize=tick_fontsize)
if xlim is not None:
    ax.set_xlim(xlim)
if ylim is not None:
    ax.set_ylim(ylim)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.18), ncol=2, fontsize=tick_fontsize, frameon=False)
ax.grid(True, alpha=0.3)
fig.tight_layout()

fig.savefig(f'{pkl_name_b}_kl_divergence_fck{fck_check_b:g}_rh{rh_check_b:g}_cov{cov_check_b:g}_{t_check_b}_pt.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
plt.show()

### 4.6 The two times side by side

The same design point at both instants, in one table and one figure.

In [13]:
stat_keys = {
                'KL':             'kl_divergence',
                'KS':             'ks_statistic',
                'Wasserstein':    'wasserstein',
                'R2 (PDF)':       'r2_pdf',
                'Rel. error P5':  'rel_err_p5',
                'Rel. error P50': 'rel_err_p50',
                'Rel. error P95': 'rel_err_p95',
            }

two_times = pd.DataFrame([
                             {'Time (years)': t_c, 'fck': fck_c, 'rh': rh_c, 'cov': cov_c,
                              **{label: res[key] for label, key in stat_keys.items()}}
                             for t_c, fck_c, rh_c, cov_c, res in [(t_check,   fck_check,   rh_check,   cov_check,   kl_result),
                                                                  (t_check_b, fck_check_b, rh_check_b, cov_check_b, kl_result_b)]
                         ])

two_times

   Time (years)       fck  ...  Rel. error P50  Rel. error P95
0          50.0  35.14612  ...       -0.015132        0.007139
1         100.0  35.14612  ...       -0.009178        0.018927

[2 rows x 11 columns]

In [14]:
fig, ax = plt.subplots(figsize=fig_size)
ax.plot(kl_result['x_grid'],   kl_result['pdf_real'],   color='0.25',    linewidth=2,                   label=f'Dataset, $t = {t_check:g}$ yr')
ax.plot(kl_result['x_grid'],   kl_result['pdf_pce'],    color='crimson', linewidth=2, linestyle='--',   label=f'PCE, $t = {t_check:g}$ yr')
ax.plot(kl_result_b['x_grid'], kl_result_b['pdf_real'], color='#2a78d6', linewidth=2,                   label=f'Dataset, $t = {t_check_b:g}$ yr')
ax.plot(kl_result_b['x_grid'], kl_result_b['pdf_pce'],  color='#eb6834', linewidth=2, linestyle='--',   label=f'PCE, $t = {t_check_b:g}$ yr')
ax.set_xlabel('$g$', fontsize=label_fontsize)
ax.set_ylabel('Probability density', fontsize=label_fontsize)
ax.tick_params(axis='both', labelsize=tick_fontsize)
if xlim is not None:
    ax.set_xlim(xlim)
if ylim is not None:
    ax.set_ylim(ylim)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.28), ncol=2, fontsize=tick_fontsize - 2, frameon=False)
ax.grid(True, alpha=0.3)
fig.tight_layout()

fig.savefig(f'{n_latent_samples}_kl_divergence_fck{fck_check:g}_rh{rh_check:g}_cov{cov_check:g}_t{t_check:g}_vs_t{t_check_b:g}_en.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
plt.show()

In [15]:
fig, ax = plt.subplots(figsize=fig_size)
ax.plot(kl_result['x_grid'],   kl_result['pdf_real'],   color='0.25',    linewidth=2,                   label=f'Dados, $t = {t_check:g}$ anos')
ax.plot(kl_result['x_grid'],   kl_result['pdf_pce'],    color='crimson', linewidth=2, linestyle='--',   label=f'PCE, $t = {t_check:g}$ anos')
ax.plot(kl_result_b['x_grid'], kl_result_b['pdf_real'], color='#2a78d6', linewidth=2,                   label=f'Dados, $t = {t_check_b:g}$ anos')
ax.plot(kl_result_b['x_grid'], kl_result_b['pdf_pce'],  color='#eb6834', linewidth=2, linestyle='--',   label=f'PCE, $t = {t_check_b:g}$ anos')
ax.set_xlabel('$g$', fontsize=label_fontsize)
ax.set_ylabel('Densidade de probabilidade', fontsize=label_fontsize)
ax.tick_params(axis='both', labelsize=tick_fontsize)
if xlim is not None:
    ax.set_xlim(xlim)
if ylim is not None:
    ax.set_ylim(ylim)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.28), ncol=2, fontsize=tick_fontsize - 2, frameon=False)
ax.grid(True, alpha=0.3)
fig.tight_layout()

fig.savefig(f'{n_latent_samples}_kl_divergence_fck{fck_check:g}_rh{rh_check:g}_cov{cov_check:g}_t{t_check:g}_vs_t{t_check_b:g}_pt.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
plt.show()

## **5. Error maps over the whole dataset**

Sweeps every design point of every time step: for each one, scores the PCE's predicted GLD against
that point's own raw Monte Carlo `g` samples, and records the seven statistics. The result is one
row per (design point, time step) — saved as a `.pkl` so it can be reused elsewhere. Commented out
by default (same as the benchmark counterpart) — a full sweep over 200 design points x 5 time steps
is not free; set `max_points_sweep` to a small number to try it out first.

In [ ]:
# lambda3_sweep = 0.132011    # same hand-written lambda 3 / lambda 4 as section 4
# lambda4_sweep = 0.141281

# split_sweep      = 'train'   # 'train' or 'val'
# n_grid_sweep     = 400
# max_points_sweep = 10        # random subsample of design points. None scores all ~200, which costs a few seconds per time step

# stat_cols = ['KL', 'KS', 'Wasserstein', 'R2 (PDF)', 'Rel. error P5', 'Rel. error P50', 'Rel. error P95']

In [ ]:
# sweep_frames = []
# for t in times:
#     tag_t = f'{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}_co2_{co2_scenario}'
#     with open(f'{n_latent_samples}_dataset_full_{split_sweep}_{tag_t}.pkl', 'rb') as f:
#         df_full_t = dill.load(f)
#     with open(f'{n_latent_samples}_pce_metamodel_{tag_t}.pkl', 'rb') as f:
#         pce_metamodel_t = dill.load(f)

#     sweep_frames.append(validate_pce_kl_divergence_dataset_durability(
#                                                                          pce_metamodel=pce_metamodel_t,
#                                                                          df_full=df_full_t,
#                                                                          time_step=t,
#                                                                          lambda3=lambda3_sweep,
#                                                                          lambda4=lambda4_sweep,
#                                                                          n_grid=n_grid_sweep,
#                                                                          max_points=max_points_sweep,
#                                                                      ))
#     del df_full_t

# pce_error_maps = pd.concat(sweep_frames, ignore_index=True)

# with open(f'{n_latent_samples}_pce_error_maps_durability.pkl', 'wb') as f:
#     dill.dump(pce_error_maps, f)

# print(f"\n{len(pce_error_maps)} rows saved to {n_latent_samples}_pce_error_maps_durability.pkl")
# pce_error_maps.head()

### 5.1 Summary per time step

In [ ]:
# pce_error_maps.groupby('Time (years)')[stat_cols].mean()